In [7]:
# 필수 라이브러리 설치 (최초 1회 실행)
import sys
!{sys.executable} -m pip install korpora pandas --quiet

In [3]:
# 필수 라이브러리 설치 (최초 1회 실행)
!pip install pandas scikit-learn boto3 transformers torch python-dotenv --quiet

## 1. 데이터 로딩 및 전처리
- 로컬 data/ 폴더에서 원본 데이터 불러오기
- 결측치/이상치 처리, 토큰화, 정제
- 훈련/검증/테스트 분할
- (Best Practice)
  - 원본 데이터는 그대로 두고, 전처리/분할된 데이터만 S3에 저장 (불필요한 중복 방지)

In [ ]:
# 환경 변수에서 AWS 자격증명 로드 (프로젝트 루트 .env.mlu 사용)
from dotenv import load_dotenv
load_dotenv('../.env.mlu')
# No secrets in code; all credentials are loaded from .env.mlu

True

In [ ]:
import os
from dotenv import load_dotenv

# Load environment variables from .env.mlu (do not print or expose secrets)
env_loaded = load_dotenv('.env.mlu')
print("dotenv loaded:", env_loaded)
# All secrets must be loaded from .env.mlu, do not print or hardcode any secret values in this notebook.
# Example usage:
# aws_access_key = os.getenv("AWS_ACCESS_KEY_ID")
# hf_token = os.getenv("HF_TOKEN")
# notion_token = os.getenv("NOTION_TOKEN")
# github_token = os.getenv("GITHUB_TOKEN")
# ...
# Do not print or log these values.
# If you previously had any secrets in this notebook, they have been removed.

dotenv loaded: False
AWS_ACCESS_KEY_ID: None
AWS_SECRET_ACCESS_KEY: None
Current working directory: d:\repos\tonylee\goorm\mlu\02_ml_basics
Files in current directory: ['.ipynb_checkpoints', '2025-11-23_fraud_detection.ipynb', 'data', 'evaluation-metrics.md', 'ml_theory_and_practice.md', 'model_df.parquet', 'movie_review_sentiment_pipeline_template.ipynb', 'movie_review_sentiment_pipeline_template.md', 'notebook_sample_inspection_cell_summary.ipynb', 'notebook_sample_inspection_cell_summary.md', 'redundant_cell_summary.md', 'test_processed.csv', 'train_processed.csv', '프로젝트010_네이버영화댓글감성분류_LSTM_BiLSTM_Attension_BERT_배포_output.ipynb']
Found: ../.env.mlu
Contents of ../.env.mlu:
 # MLU Docker Environment Configuration
JUPYTER_TOKEN=mlu2025
COMPOSE_PROJECT_NAME=mlu
DOCKER_BUILDKIT=1

# --- Imported from ai-track/.env ---
HF_TOKEN=your_huggingface_token_here

# Notion API Configuration
NOTION_TOKEN=your_notion_token_here
NOTION_API_KEY=your_notion_token_here
NOTION_QUIZ_DB_ID=2a48897f-40c2-

In [ ]:
import os
import pandas as pd
import boto3
# 데이터 로딩 (이미 전처리된 데이터 사용)
data_path = './data/preprocessed.csv'
df = pd.read_csv(data_path)
# 데이터 분할 (예시)
from sklearn.model_selection import train_test_split
train, test = train_test_split(df, test_size=0.2, random_state=42)
# S3에 저장 (Best Practice: 처리된 데이터만)
s3 = boto3.client('s3')
bucket_name = os.getenv('S3_BUCKET_NAME', 'elbee-oreumi')  # 실제 S3 버킷명은 .env.mlu에서 관리
try:
    train.to_csv('train_processed.csv', index=False)
    test.to_csv('test_processed.csv', index=False)
    # S3 upload using credentials from .env.mlu only
    s3.upload_file('train_processed.csv', bucket_name, 'processed/train_processed.csv')
    s3.upload_file('test_processed.csv', bucket_name, 'processed/test_processed.csv')
except Exception as e:
    print(f'⚠️ S3 업로드 오류: {e}\n버킷 이름, 권한, 네트워크를 확인하세요.')
# No secrets are present in this cell or output.

c:\Users\hsyyu\anaconda3\envs\mlu\lib\site-packages\boto3\compat.py:84: PythonDeprecationWarning: Boto3 will no longer support Python 3.9 starting April 29, 2026. To continue receiving service updates, bug fixes, and security updates please upgrade to Python 3.10 or later. More information can be found here: https://aws.amazon.com/blogs/developer/python-support-policy-updates-for-aws-sdks-and-tools/
  warnings.warn(warning, PythonDeprecationWarning)


In [ ]:
# Naver Movie Review Data Download & Exploration with Korpora (robust, with pip subprocess and sys.path reload workaround)

import sys
import importlib
import pandas as pd
import os

def install_and_import(package):
    try:
        return importlib.import_module(package)
    except ImportError:
        import subprocess
        print(f"[INFO] {package} not found. Installing...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", package])
        # Try to reload sys.path in case Jupyter doesn't see new packages
        import site
        import importlib
        importlib.reload(site)
        if hasattr(sys, 'path_importer_cache'):
            sys.path_importer_cache.clear()
        return importlib.import_module(package)

try:
    korpora = install_and_import('korpora')
    Korpora = getattr(korpora, 'Korpora')
    dataset = Korpora.load('naver_movie')
except Exception as e:
    print(f"[ERROR] Could not import or load korpora: {e}")
    print("Falling back to local ratings_train.txt and ratings_test.txt")
    train = pd.read_csv('./data/ratings_train.txt', sep='\t')
    test = pd.read_csv('./data/ratings_test.txt', sep='\t')
else:
    train = pd.DataFrame({
        'id': [x.id for x in dataset.train],
        'document': [x.text for x in dataset.train],
        'label': [int(x.label) for x in dataset.train]
    })
    test = pd.DataFrame({
        'id': [x.id for x in dataset.test],
        'document': [x.text for x in dataset.test],
        'label': [int(x.label) for x in dataset.test]
    })

os.makedirs('data', exist_ok=True)
train.to_csv('data/ratings_train_korpora.csv', index=False)
test.to_csv('data/ratings_test_korpora.csv', index=False)
print(f"Train shape: {train.shape}")
print(f"Test shape: {test.shape}")
print(train.head())
# No secrets in code or output; all credentials are loaded from .env.mlu

[ERROR] Could not import or load korpora: local variable 'importlib' referenced before assignment
Falling back to local ratings_train.txt and ratings_test.txt
Train shape: (150000, 3)
Test shape: (50000, 3)
         id                                           document  label
0   9976970                                아 더빙.. 진짜 짜증나네요 목소리      0
1   3819312                  흠...포스터보고 초딩영화줄....오버연기조차 가볍지 않구나      1
2  10265843                                  너무재밓었다그래서보는것을추천한다      0
3   9045019                      교도소 이야기구먼 ..솔직히 재미는 없다..평점 조정      0
4   6483659  사이몬페그의 익살스런 연기가 돋보였던 영화!스파이더맨에서 늙어보이기만 했던 커스틴 ...      1
Train shape: (150000, 3)
Test shape: (50000, 3)
         id                                           document  label
0   9976970                                아 더빙.. 진짜 짜증나네요 목소리      0
1   3819312                  흠...포스터보고 초딩영화줄....오버연기조차 가볍지 않구나      1
2  10265843                                  너무재밓었다그래서보는것을추천한다      0
3   9045019                      교도소 이야기구먼 ..

### 데이터 파이프라인 설명 및 AWS 활용 방식

#### 용어 및 메서드 정의

- **DataFrame**: pandas의 2차원 데이터 구조로, 표 형태의 데이터를 다룸.
- **train_test_split**: scikit-learn의 함수로, 데이터를 훈련/테스트 세트로 분할.
- **to_csv**: pandas DataFrame을 CSV 파일로 저장하는 메서드.
- **boto3.client('s3')**: AWS S3와 상호작용하는 파이썬 라이브러리의 클라이언트 객체 생성.
- **upload_file**: boto3의 S3 클라이언트 메서드로, 로컬 파일을 S3 버킷에 업로드.

#### 파이프라인 동작 요약

1. 전처리된 데이터를 불러와 DataFrame으로 로드합니다.
2. 데이터를 훈련/테스트 세트로 분할합니다.
3. 분할된 데이터를 각각 CSV 파일로 저장합니다.
4. 이 파일들을 AWS S3 버킷의 `processed/` 폴더에 업로드합니다.

#### AWS 파이프라인에서의 활용
- 이 방식은 **데이터 준비 및 업로드 자동화**의 기초 단계입니다.
- S3에 업로드된 데이터는 이후 AWS의 다양한 서비스(예: SageMaker, Lambda, Glue, EMR 등)에서 바로 접근하여 사용할 수 있습니다.
- 예시: SageMaker에서 학습 작업을 생성할 때, S3의 `processed/train_processed.csv`와 `processed/test_processed.csv` 경로를 입력하면 별도의 데이터 이동 없이 바로 학습에 활용할 수 있습니다.
- 이 구조는 **데이터 파이프라인의 표준화**와 **재현성**을 높여주며, 협업 및 자동화된 MLOps 환경 구축의 기반이 됩니다.

In [16]:
# S3 버킷에 저장된 파일 목록 확인
try:
    response = s3.list_objects_v2(Bucket=bucket_name, Prefix='processed/')
    if 'Contents' in response:
        print('S3 버킷 내 processed/ 폴더 파일 목록:')
        for obj in response['Contents']:
            print(obj['Key'], f"({obj['Size']} bytes)")
    else:
        print('S3 버킷에 processed/ 폴더가 비어있거나 파일이 없습니다.')
except Exception as e:
    print(f'⚠️ S3 목록 조회 오류: {e}')

S3 버킷 내 processed/ 폴더 파일 목록:
processed/test_processed.csv (663384 bytes)
processed/train_processed.csv (2647089 bytes)


In [6]:
import sys
print(sys.executable)
!which python
!pip show korpora

c:\Users\hsyyu\anaconda3\envs\mlu\python.exe


'which' is not recognized as an internal or external command,
operable program or batch file.


Name: Korpora
Version: 0.2.0
Summary: This package provides easy-download and easy-usage for various Korean corpora.
Home-page: https://github.com/ko-nlp/Korpora
Author: ko-nlp
Author-email: 
License: UNKNOWN
Location: c:\users\hsyyu\anaconda3\envs\mlu\lib\site-packages
Requires: dataclasses, numpy, requests, tqdm, xlrd
Required-by: 
